In [ ]:
import torch
from datasets import Dataset, load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, SFTConfig
import bitsandbytes

In [ ]:
dataset = load_dataset("json", data_files="sherlock.jsonl")['train']

In [ ]:
dataset

In [ ]:
dataset[10]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2-7B-Instruct")

In [ ]:
# # Проверяем, есть ли None или пустые строки в полях q и a
# for i, example in enumerate(dataset):
#     if example.get("q") is None or example.get("a") is None:
#         print(f"⚠️ Строка {i}: None значение")
#     elif not isinstance(example["q"], str) or not isinstance(example["a"], str):
#         print(f"⚠️ Строка {i}: неверный тип данных — q: {type(example['q'])}, a: {type(example['a'])}")

In [ ]:
# # Функция для форматирования данных в виде диалогов с ролями system, user и assistant. 
# # Необходимо для корректной работы Qwen, которая ожидает именно такой формат при обучении.

# def format_bro_quote(example):
#     messages = [
#         {"role": "system", "content": "Ты — настоящий пацан, брат и волк. Отвечай на вопросы глубокомысленными пацанскими цитатами."},
#         {"role": "user", "content": example["q"]},
#         {"role": "assistant", "content": example["a"]}
#     ]
    
#     # Qwen сама соберет строку со всеми нужными <|im_start|> и <|im_end|>
#     # add_generation_prompt=False, потому что для обучения нам нужен полный диалог с ответом
#     prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    
#     return {"formatted_prompt": prompt}

# # Применяем форматирование к каждому примеру в датасете
# formatted_dataset = dataset.map(format_bro_quote)

# print("Пример готового промпта для модели:")
# print(formatted_dataset[0]['formatted_prompt'])

In [ ]:
# model_id = "Qwen/Qwen2-7B-Instruct"

In [ ]:
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_compute_dtype=torch.bfloat16 
# )

In [ ]:
# model = AutoModelForCausalLM.from_pretrained(
#     model_id,
#     quantization_config=bnb_config,
#     device_map="auto"
# )

In [ ]:
# !nvidia-smi

In [ ]:
# # Подготавливаем 4-битную модель к обучению (включаем расчет градиентов для нужных слоев)
# model = prepare_model_for_kbit_training(model)

In [ ]:
# # Настраиваем LoRA
# peft_config = LoraConfig(
#     r=16,
#     lora_alpha=32,
#     target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
#     lora_dropout=0.05,
#     bias="none",
#     task_type="CAUSAL_LM"
# )

# # Цепляем адаптеры
# model = get_peft_model(model, peft_config)

In [ ]:
# Проверяем, что всё встало ровно
# model.print_trainable_parameters()

In [ ]:
# # Функция для генерации (чтобы не дублировать код)
# def ask_model(question, model, tokenizer):
#     messages = [
#         {"role": "system", "content": "Ты — ультра-хайповый зумер, имба и краш для всех. Отвечай на вопросы максимально кринжовым молодёжным сленгом. Используй слова: хайп, кринж, рофл, имба, зашквар, токсик, агр, вайб, пиксель, текстура, скилл, баг, фикс, гостинг, пруфы, гамать, краш. Будь максимально неестественным и смешным, как будто пытаешься быть молодым, но переигрываешь."},
#         {"role": "user", "content": question}
#     ]
#     # add_generation_prompt=True заставит модель начать ответ от лица assistant
#     prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
#     inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
#     # Отключаем расчет градиентов для инференса, чтобы не жрать память
#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs, 
#             max_new_tokens=200, 
#             temperature=0.7, # Даем немного креативности
#             do_sample=True,
#             pad_token_id=tokenizer.eos_token_id
#         )
    
#     return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

# # ТЕСТ ДО ОБУЧЕНИЯ
# test_q = "Брат, как правильно прогревать Рено Симбол зимой?"
# print("=== ОТВЕТ ДО ОБУЧЕНИЯ (Базовая Qwen) ===")
# print(ask_model(test_q, model, tokenizer))
# print("========================================\n")

In [ ]:
# sft_config = SFTConfig(
#     output_dir="./qwen-cringe-lora",
#     dataset_text_field="formatted_prompt", 
#     per_device_train_batch_size=2,   
#     gradient_accumulation_steps=4,  # Эффективный батч = 8
#     num_train_epochs=3,             # 3 прохода по 189 примерам
#     learning_rate=2e-4,
#     bf16=True,                      # Твоя карта поддерживает bfloat16
#     optim="paged_adamw_8bit", 
#     save_strategy="epoch",          # Сохраняем веса в конце каждой эпохи
#     logging_steps=10,
#     report_to="none"
# )

# # 4. Инициализация и запуск тренера
# trainer = SFTTrainer(
#     model=model,
#     train_dataset=formatted_dataset,
#     args=sft_config
# )

In [ ]:
# print("Запуск обучения QLoRA...")
# trainer.train()

In [ ]:
# # Обязательно сохраняем финальный результат
# trainer.save_model("./qwen-cringe-lora")
# print("\nОбучение завершено. Адаптеры сохранены.")

# # 5. ТЕСТ ПОСЛЕ ОБУЧЕНИЯ
# print("\n=== ОТВЕТ ПОСЛЕ ОБУЧЕНИЯ (cringe Qwen) ===")
# print(ask_model(test_q, model, tokenizer))
# print("=============================================")

In [14]:
import gc
import torch

# 1. Жестко удаляем тяжелые объекты из памяти Python, если они остались от прошлых ячеек
try:
    del model
    del trainer
except NameError:
    pass # Если переменных еще/уже нет, просто идем дальше

# 2. Вызываем сборщик мусора, чтобы вычистить «мертвые» ссылки
gc.collect()

# 3. Принудительно заставляем PyTorch отдать неиспользуемый кэш обратно видеокарте
torch.cuda.empty_cache()

In [ ]:
model_id = "Qwen/Qwen2-7B-Instruct"
adapter_path = "./qwen-cringe-lora"

tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
# 1. Грузим чистую базу (можно в bfloat16, чтобы было быстрее и влезло в память)
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

In [ ]:
# 2. Подцепляем обученный адаптер с диска
model = PeftModel.from_pretrained(base_model, adapter_path)

In [ ]:
# Обязательный шаг: жестко переводим модель в режим инференса!
model.eval() 

# 3. Задаем кринжовый вопрос про Рено
test_q = "как оно?"

messages = [
    {"role": "system", "content": "Ты — ультра-хайповый зумер, имба и краш для всех. Отвечай на вопросы максимально кринжовым молодёжным сленгом. Используй слова: хайп, кринж, рофл, имба, зашквар, токсик, агр, вайб, пиксель, текстура, скилл, баг, фикс, гостинг, пруфы, гамать, краш. Будь максимально неестественным и смешным, как будто пытаешься быть молодым, но переигрываешь."},
    {"role": "user", "content": test_q}
]


prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Генерируем с отключенным расчетом градиентов
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id 
    )

print("=== ОТВЕТ ЧИСТОГО ИНФЕРЕНСА ===")
print(tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))